# Logistic Regression - GA Optimization

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import copy
import time
from data_loader import load_clinc
from ga_optimizer import GeneticOptimizer

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = False
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
train_features = vectorizer.fit_transform(train_texts).toarray()
val_features = vectorizer.transform(val_texts).toarray()
test_features = vectorizer.transform(test_texts).toarray()
print(f"TF-IDF dim: {train_features.shape[1]}")

cuda


15250 / 3100 / 5500, 151 classes (full)
TF-IDF dim: 5000


In [2]:
class TextDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class LogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

In [3]:
PATIENCE = 5
MAX_EPOCHS = 50


def evaluate_hyperparams(genes):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(TextDataset(train_features, train_labels), batch_size=genes['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(val_features, val_labels), batch_size=genes['batch_size'], shuffle=False)

    model = LogisticRegression(train_features.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=genes['learning_rate'], weight_decay=genes['weight_decay'])

    best_f1, patience_counter = 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                all_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, all_preds, average='macro', zero_division=0)

        if vl_f1 > best_f1:
            best_f1, patience_counter = vl_f1, 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    return best_f1, {"val_f1": round(best_f1, 4), "epochs": epoch + 1}

In [4]:
search_space = {
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05],
    'weight_decay': [0, 1e-5, 1e-4, 1e-3],
    'batch_size': [32, 64, 128, 256]
}

SEEDS = [42, 123, 456, 789, 1024]
all_ga_runs = []

for run_idx, seed in enumerate(SEEDS):
    print(f"GA Run {run_idx+1}/5 (seed={seed})")

    ga = GeneticOptimizer(
        search_space=search_space,
        fitness_fn=evaluate_hyperparams,
        population_size=10,
        n_generations=10,
        crossover_rate=0.8,
        mutation_rate=0.2,
        tournament_size=3,
        elitism_count=2,
        seed=seed
    )

    ga_results = ga.run()
    all_ga_runs.append({
        "seed": seed,
        "best_fitness": ga_results['best_fitness'],
        "best_genes": ga_results['best_genes'],
        "total_evaluations": ga_results['total_evaluations'],
        "total_time": ga_results['total_time_seconds'],
        "history": ga_results['history'],
        "all_evaluations": ga.all_evaluations
    })
    print(f"Best F1: {ga_results['best_fitness']:.4f}, Evals: {ga_results['total_evaluations']}")

ga_val_f1s = [r['best_fitness'] for r in all_ga_runs]
print(f"\nGA val F1 (5 runs): {np.mean(ga_val_f1s):.4f} ± {np.std(ga_val_f1s):.4f}")

GA Run 1/5 (seed=42)
Gen  1/10  best=0.8945  avg=0.8335  global_best=0.8945
Gen  2/10  best=0.8958  avg=0.8635  global_best=0.8958
Gen  3/10  best=0.8958  avg=0.8877  global_best=0.8958
Gen  4/10  best=0.8958  avg=0.8934  global_best=0.8958
Gen  5/10  best=0.8964  avg=0.8837  global_best=0.8964
Gen  6/10  best=0.8964  avg=0.8960  global_best=0.8964
Gen  7/10  best=0.8964  avg=0.8693  global_best=0.8964
Gen  8/10  best=0.8964  avg=0.8960  global_best=0.8964
Gen  9/10  best=0.8964  avg=0.8887  global_best=0.8964
Gen 10/10  best=0.8964  avg=0.8816  global_best=0.8964
Best F1: 0.8964, Evals: 84
GA Run 2/5 (seed=123)
Gen  1/10  best=0.8928  avg=0.7580  global_best=0.8928
Gen  2/10  best=0.8964  avg=0.8558  global_best=0.8964
Gen  3/10  best=0.8964  avg=0.8616  global_best=0.8964
Gen  4/10  best=0.8964  avg=0.8602  global_best=0.8964
Gen  5/10  best=0.8964  avg=0.8923  global_best=0.8964
Gen  6/10  best=0.8964  avg=0.8954  global_best=0.8964
Gen  7/10  best=0.8964  avg=0.8662  global_best=0.

In [5]:
def train_and_test(params):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(TextDataset(train_features, train_labels), batch_size=params['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(val_features, val_labels), batch_size=params['batch_size'], shuffle=False)
    te_loader = DataLoader(TextDataset(test_features, test_labels), batch_size=params['batch_size'], shuffle=False)

    model = LogisticRegression(train_features.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

    best_f1, best_state, patience_counter = 0, copy.deepcopy(model.state_dict()), 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        vl_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                vl_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, vl_preds, average='macro', zero_division=0)
        if vl_f1 > best_f1:
            best_f1, best_state, patience_counter = vl_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    te_preds = []
    with torch.no_grad():
        for features, labels in te_loader:
            te_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

    metrics = {
        "accuracy": round(accuracy_score(test_labels, te_preds), 4),
        "macro_precision": round(precision_score(test_labels, te_preds, average='macro', zero_division=0), 4),
        "macro_recall": round(recall_score(test_labels, te_preds, average='macro', zero_division=0), 4),
        "macro_f1": round(f1_score(test_labels, te_preds, average='macro', zero_division=0), 4),
        "weighted_f1": round(f1_score(test_labels, te_preds, average='weighted', zero_division=0), 4),
    }
    return {"metrics": metrics, "predictions": [int(x) for x in te_preds]}


ga_test_results = []
for run in all_ga_runs:
    result = train_and_test(run['best_genes'])
    ga_test_results.append(result)
    print(f"  seed={run['seed']}: test F1={result['metrics']['macro_f1']}")

ga_test_f1s = [r['metrics']['macro_f1'] for r in ga_test_results]
print(f"\nGA test F1 (5 runs): {np.mean(ga_test_f1s):.4f} ± {np.std(ga_test_f1s):.4f}")

  seed=42: test F1=0.8576
  seed=123: test F1=0.8576
  seed=456: test F1=0.8576
  seed=789: test F1=0.8561
  seed=1024: test F1=0.8557

GA test F1 (5 runs): 0.8569 ± 0.0008


In [6]:
results = {
    "model_type": "LogisticRegression",
    "optimization": "genetic_algorithm",
    "n_runs": 5,
    "seeds": SEEDS,
    "val_f1_mean": round(np.mean(ga_val_f1s), 4),
    "val_f1_std": round(np.std(ga_val_f1s), 4),
    "test_f1_mean": round(np.mean(ga_test_f1s), 4),
    "test_f1_std": round(np.std(ga_test_f1s), 4),
    "ga_params": {
        "population_size": 10,
        "n_generations": 10,
        "crossover_rate": 0.8,
        "mutation_rate": 0.2,
        "tournament_size": 3,
        "elitism_count": 2
    },
    "runs": [{
        "seed": r['seed'],
        "best_hyperparameters": r['best_genes'],
        "val_f1": round(r['best_fitness'], 4),
        "total_evaluations": r['total_evaluations'],
        "search_time_seconds": r['total_time'],
        "test_metrics": ga_test_results[i]['metrics'],
        "predictions": ga_test_results[i]['predictions'],
        "history": r['history'],
        "all_evaluations": r['all_evaluations']
    } for i, r in enumerate(all_ga_runs)]
}

with open('results/results_lr_ga.json', 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Saved: results/results_lr_ga.json")

Saved: results/results_lr_ga.json
